# Assembler des séquences de durées différentes

Construire un lot sans perdre les longueurs réelles.

Nous exécutons les cellules dans l’ordre, sur **CPU**. Aucun téléchargement de données ou de modèle n’est nécessaire.

Version de référence : **Python 3.11, PyTorch 2.10.0+cpu**. La cellule suivante indique la version réellement utilisée. Les sorties peuvent varier avec une autre version.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#dataset-et-batches). Les deux exercices sont des textes à compléter : ils n’interrompent pas l’exécution intégrale. Leurs solutions restent repliées à la fin.

In [1]:
import platform
import torch

torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", platform.python_version())
print("PyTorch", torch.__version__)
print("Calcul", "CPU", "— threads", torch.get_num_threads())

Python 3.11.9
PyTorch 2.10.0+cpu
Calcul CPU — threads 1


## Nos repères

- **Jeu de données (Dataset)** : La manière de retrouver un exemple annoté.
- **Chargeur (DataLoader)** : L’organisation du parcours et des lots.
- **Assemblage (collate_fn)** : La fonction qui réunit une liste d’exemples.
- **Remplissage (padding)** : Des valeurs ajoutées pour obtenir des tailles compatibles.

## Le calcul du cours

Avant de lancer cette cellule, annonçons les entrées et la sortie attendue. Après l’exécution, expliquons les nombres et les formes affichés. Le programme est identique à celui du chapitre.

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class Extraits(Dataset):
    def __init__(self):
        self.signaux = [torch.tensor([[0., 1., 2.], [2., 3., 4.]]),
                        torch.tensor([[2., 0., 1.]])]
        self.cibles = [0, 1]
    def __len__(self):
        return len(self.signaux)
    def __getitem__(self, indice):
        return self.signaux[indice], self.cibles[indice]

def assembler(exemples):
    signaux, cibles = zip(*exemples)
    longueurs = torch.tensor([len(x) for x in signaux])
    valeurs = pad_sequence(signaux, batch_first=True)
    masque = torch.arange(valeurs.shape[1])[None, :] < longueurs[:, None]
    return valeurs, torch.tensor(cibles), longueurs, masque

loader = DataLoader(Extraits(), batch_size=2, collate_fn=assembler,
                    shuffle=False, num_workers=0)
batch, cibles, longueurs, masque = next(iter(loader))
print("forme", tuple(batch.shape))
print("cibles", cibles.tolist(), "longueurs", longueurs.tolist())
print("masque", masque.tolist())

forme (2, 2, 3)
cibles [0, 1] longueurs [2, 1]
masque [[True, True], [True, False]]


## Nous vérifions notre compréhension

À dire en entretien : Dataset définit un exemple et DataLoader organise son parcours. Ma collate_fn produit le lot, ses cibles et ses longueurs, sans confondre remplissage et observation.

## Exercice 1 — Décrire un jeu de données

Complétons les deux méthodes qui permettent au chargeur de retrouver un exemple et sa cible.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
from torch.utils.data import Dataset
class Mini(Dataset):
    def __init__(self):
        self.exemples = [torch.tensor([1., 2.]), torch.tensor([3., 4.])]
        self.cibles = [0, 1]
    def __len__(self):
        return ...
    def __getitem__(self, i):
        return ..., ...
jeu = Mini()
print(len(jeu), jeu[1])
```

## Exercice 2 — Assembler des durées différentes

Complétons le remplissage et le masque. Une valeur ajoutée pour remplir ne devient pas une observation réelle.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
from torch.nn.utils.rnn import pad_sequence
def assembler(exemples):
    signaux, cibles = zip(*exemples)
    longueurs = torch.tensor([len(x) for x in signaux])
    valeurs = pad_sequence(signaux, batch_first=...)
    masque = torch.arange(valeurs.shape[1])[None, :] < ...
    return valeurs, torch.tensor(cibles), masque
lot = [(torch.tensor([[1.], [2.]]), 0), (torch.tensor([[3.]]), 1)]
valeurs, cibles, masque = assembler(lot)
print(tuple(valeurs.shape), masque.tolist())
```

# Solutions

Nous cherchons d’abord, puis ouvrons la solution utile. Chaque programme ci-dessous est autonome. Le vérificateur du dépôt exécute également ces solutions, séparément des exercices à trous.

<details>
<summary>Afficher la solution de l’exercice 1 — Décrire un jeu de données</summary>

<pre><code>import torch
from torch.utils.data import Dataset
class Mini(Dataset):
    def __init__(self):
        self.exemples = [torch.tensor([1., 2.]), torch.tensor([3., 4.])]
        self.cibles = [0, 1]
    def __len__(self):
        return len(self.exemples)
    def __getitem__(self, i):
        return self.exemples[i], self.cibles[i]
jeu = Mini()
assert len(jeu) == 2 and jeu[1][1] == 1
print(len(jeu), jeu[1])</code></pre>
</details>

<details>
<summary>Afficher la solution de l’exercice 2 — Assembler des durées différentes</summary>

<pre><code>import torch
from torch.nn.utils.rnn import pad_sequence
def assembler(exemples):
    signaux, cibles = zip(*exemples)
    longueurs = torch.tensor([len(x) for x in signaux])
    valeurs = pad_sequence(signaux, batch_first=True)
    masque = torch.arange(valeurs.shape[1])[None, :] &lt; longueurs[:, None]
    return valeurs, torch.tensor(cibles), masque
lot = [(torch.tensor([[1.], [2.]]), 0), (torch.tensor([[3.]]), 1)]
valeurs, cibles, masque = assembler(lot)
assert tuple(valeurs.shape) == (2, 2, 1)
assert masque.tolist() == [[True, True], [True, False]]
assert cibles.tolist() == [0, 1]
print(tuple(valeurs.shape), masque.tolist())</code></pre>
</details>